My solution for the titanic ml challenge

First I load training data from csv to data frame

In [2]:
import pandas as pd

df = pd.read_csv("./titanic/train.csv")

#df.shape
display(df.head(5))

#print("Survived females: ", ((df["Sex"] == "female") & (df["Survived"] == 1)).sum())
#print("Survived males: ", ((df["Sex"] == "male") & (df["Survived"] == 1)).sum())

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Some data exploration, seeing how classes affected survivability and who exactly survived

In [3]:
data = []
for pclass in sorted(df["Pclass"].unique()):
    total = (df["Pclass"] == pclass).sum()
    total_males = ((df["Sex"] == "male") & (df["Pclass"] == pclass)).sum()
    total_females = ((df["Sex"] == "female") & (df["Pclass"] == pclass)).sum()

    survived = ((df["Pclass"] == pclass) & (df["Survived"] == 1))
    men_survived = ((df["Sex"] == "male") & survived).sum()
    female_survived = ((df["Sex"] == "female") & survived).sum()

    kids = ((df["Age"] < 18) & survived).sum()

    data.append({
        "Class": pclass,
        "Passengers": total,
        "Survived": survived.sum(),
        "Survived children": kids, 
        "Percentage (%)": survived.sum() / total * 100,
        "Survived men": men_survived,
        "Total men": total_males,
        "Survived females": female_survived,
        "Total Females": total_females
    })

summary = pd.DataFrame(data)
display(summary.style
        .format({"Percentage (%)": "{:.2f}%"})
        .hide(axis="index"))

Class,Passengers,Survived,Survived children,Percentage (%),Survived men,Total men,Survived females,Total Females
1,216,136,11,62.96%,45,122,91,94
2,184,87,21,47.28%,17,108,70,76
3,491,119,29,24.24%,47,347,72,144


Missing values

In [4]:
#df.shape

df.info()

print(df["Cabin"].isna().sum())
print(df["Age"].isna().sum())
print(df["Embarked"].isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB
687
177
2


Missing values:
- cabin (687)
- age (177)
- Embarked (2)

I will skip Cabin column entirely for now, in future i will try to make it known/unknown and see if model improves

For age i will imputate the missing values with median age and also test model with age - known/unknown feature

Embarked - I will just fill this with the most popular port of embarkation present in dataset

Data will be filled based on training groups

Splitting data into training and testing split

In [5]:
from typing import cast

from pandas import DataFrame, Series
from sklearn.model_selection import train_test_split


X = df.drop(columns=["Cabin", "Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = cast(
    tuple[DataFrame, DataFrame, Series, Series],
    train_test_split(X, y, test_size=0.2, random_state=42, stratify=y),
)

print(X_train.shape)
print(X_test.shape)

(712, 10)
(179, 10)


Baseline - dummy classifier with most_frequent strategy so it will just spit the most frequent class in our dataset as its prediction. This is my baseline model which i will compare to my Logistic Regression

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

dummy = DummyClassifier(strategy="most_frequent")

dummy.fit(X_train, y_train)

print("Classes:", dummy.classes_)
print("Class distribution:")
print(y_train.value_counts(normalize=True))

y_train_pred = dummy.predict(X_train)

accuracy = accuracy_score(y_train, y_train_pred)
precision = precision_score(y_train, y_train_pred, zero_division=0)
recall = recall_score(y_train, y_train_pred)
f1 = f1_score(y_train, y_train_pred)

print("Accuracy: ", accuracy)
print("Precision: ", precision)
print("Recall : ", recall)
print("F1: ", f1)

Classes: [0 1]
Class distribution:
Survived
0    0.616573
1    0.383427
Name: proportion, dtype: float64
Accuracy:  0.6165730337078652
Precision:  0.0
Recall :  0.0
F1:  0.0


Cross validation to see how our dummy performs on different data

In [7]:
from sklearn.model_selection import cross_val_score


scores = cross_val_score(dummy, X_train, y_train, cv=5, scoring="accuracy")
print("CV accuracy scores: ", scores)
print("CV mean accuracy: ", scores.mean())

CV accuracy scores:  [0.61538462 0.61538462 0.61267606 0.61971831 0.61971831]
CV mean accuracy:  0.6165763813651137


Now I want to classify my features into numeric and categorical features because I want to preprocess them first to fill blank values 

In [8]:
numeric_features = ["Age", "Fare", "SibSp", "Parch"]

categorical_features = ["Pclass", "Sex", "Embarked"]

features = numeric_features + categorical_features

X_train_selected = X_train[features]

#X_train_selected.shape
X_train_selected.head(5)

,Age,Fare,SibSp,Parch,Pclass,Sex,Embarked
692,NaN,56.4958,0,0,3,male,S
481,NaN,0.0000,0,0,2,male,S
527,NaN,221.7792,0,0,1,male,S
855,18.0,9.3500,0,1,3,female,S
801,31.0,26.2500,1,1,2,female,S


Now i want to build a pipeline which will transform my data into desired format.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

Now  I can create my model using my data transformer pipeline

In [10]:
from sklearn.linear_model import LogisticRegression


model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

Lets train it!

In [11]:
logistic_scores = cross_val_score(model, X_train_selected, y_train, cv=5, scoring="accuracy")

print(logistic_scores)
print("Mean accuracy: ", logistic_scores.mean())

[0.7972028  0.76223776 0.8028169  0.80985915 0.81690141]
Mean accuracy:  0.7978036048458584


#### Model comparison with cross validation

Comparing DummyClassifier, Logistic Regression, Decision Tree and Random Forest using the same folds, preprocessing and input features.

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

# Defining our data split strategy, data will be shuffled before splitted into 5 folds
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

models = {
  "DummyClassifier": dummy,
  "LogisticRegression": LogisticRegression(max_iter=1000),
  "DecisionTree": DecisionTreeClassifier(
    max_depth=3,
    random_state=42
  ),
  "RandomForest": RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
  )
}

Now i write a loop which will compare different classificators

In [13]:
from sklearn.metrics import make_scorer


results = []

for name, classifier in models.items():
    model_pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("classifier", classifier)
    ])

    cv_scores = cross_validate(
        model_pipeline,
        X_train_selected,
        y_train,
        cv=cv,
        scoring= {
            "accuracy": "accuracy",
            "precision": make_scorer(
                precision_score,
                zero_division=0
            ),
            "recall": "recall",
            "f1": "f1" 
        }
    )

    results.append({
        "Model": name,
        "Accuracy": cv_scores["test_accuracy"].mean(),
        "Precision": cv_scores["test_precision"].mean(),
        "Recall": cv_scores["test_recall"].mean(),
        "F1": cv_scores["test_f1"].mean()
    })


results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="Accuracy",
    ascending=False
)

display(results_df.round(4))

,Model,Accuracy,Precision,Recall,F1
3,RandomForest,0.8160,0.8177,0.6700,0.7355
2,DecisionTree,0.8062,0.8142,0.6484,0.7180
1,LogisticRegression,0.8020,0.7652,0.6997,0.7301
0,DummyClassifier,0.6166,0.0000,0.0000,0.0000


#### Random forest Hyperparemeter tuning

In [17]:
from sklearn.model_selection import GridSearchCV

# Pipeline
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

# Our options we want to test
param_grid = {
    "classifier__max_depth": [3, 5, 7, None],
    "classifier__min_samples_leaf": [1, 3, 5]
}

# GridSearchCV will try all combinations of options and get the one with best accuracy
grid_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train_selected, y_train)

print("Best parameters: ", grid_search.best_params_)
print("Best CV accuracy: ", grid_search.best_score_)

results = pd.DataFrame(grid_search.cv_results_)

display(
    results[[
        "param_classifier__max_depth",
        "param_classifier__min_samples_leaf",
        "mean_test_score",
        "mean_train_score",
        "std_test_score",
        "rank_test_score"
    ]].sort_values("rank_test_score")
)

Best parameters:  {'classifier__max_depth': 7, 'classifier__min_samples_leaf': 1}
Best CV accuracy:  0.821619225844578


,param_classifier__max_depth,param_classifier__min_samples_leaf,mean_test_score,mean_train_score,std_test_score,rank_test_score
6,7,1,0.821619,0.908711,0.030383,1
10,None,3,0.817414,0.894317,0.025115,2
4,5,3,0.815985,0.855336,0.024683,3
3,5,1,0.815976,0.862361,0.022673,4
0,3,1,0.814656,0.819172,0.032225,5
5,5,5,0.814577,0.850772,0.026343,6
8,7,5,0.813188,0.869383,0.027352,7
7,7,3,0.811799,0.876758,0.031533,8
1,3,3,0.810450,0.816714,0.029424,9
11,None,5,0.807594,0.873600,0.023678,10


#### Making predictions on test data with the best model

In [18]:
best_model = grid_search.best_estimator_

X_test_selected = X_test[features]

y_test_pred = best_model.predict(X_test_selected)

accuracy = accuracy_score(y_test, y_test_pred)
precision = precision_score(y_test, y_test_pred)
recall_score = recall_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred)

print("Test accuracy: ", accuracy)
print("Test precision: ", precision)
print("Test recall: ", recall_score)
print("Test F1: ", f1)


Test accuracy:  0.7988826815642458
Test precision:  0.8367346938775511
Test recall:  0.5942028985507246
Test F1:  0.6949152542372882
